# 1-D Energy Balance Model (EBM) with Diffusive Heat Transport

**Theoretical Background:** [`Docs/EBM_eng3.pdf`](Docs/EBM_eng3.pdf)  
**Author:** Halldór Björnsson  

---

## Overview

This notebook provides an interactive environment to explore the **1-D diffusive energy balance model** (North 1975, JAS 32) and its associated climate dynamics:

1. **Model Formulation:** Steady-state thermal balance across latitude $\theta$ (using $x = \sin\theta \in [0, 1]$):
   $$Q_{sw}(x) + Q_T(x) - Q_{lw}(x) = 0$$
   where:
   - Absorbed shortwave: $Q_{sw}(x) = \frac{S_0}{4} (1 - 0.492 P_2(x)) (1 - \alpha(x))$
   - Outgoing longwave: $Q_{lw}(x) = A + B T(x)$
   - Diffusive poleward heat transport: $Q_T(x) = \frac{d}{dx} \left[ D(x) (1 - x^2) \frac{dT}{dx} \right]$
   - Boundary conditions: $F(0) = 0$ (equatorial symmetry), $F(1) = 0$ (polar boundary).

2. **Exact Analytical Solution:** Comparison with the 2-term Legendre polynomial solution:
   $$T(x) = \frac{a Q - A}{B} + \frac{\beta a Q}{6D + B} P_2(x)$$

3. **Assignment 1 Experiments:** Sensitivity to diffusivity $D$, latitude-dependent albedo $\alpha(x)$, and Hadley-cell enhanced diffusivity $D(x)$ (Lindzen & Farrell 1977) compared against NCEP reanalysis data.

4. **Small Ice Cap Instability (SICI):** Temperature-albedo feedback ($\alpha = 0.60$ for $T < -10^\circ\mathrm{C}$, else $0.33$) and hysteresis under varying solar forcing $S_0$ (Assignment 2).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import model components from ebm_model, run_ebm, and ebm_sici
from ebm_model import EnergyBalanceModel, legendre2
from ebm_sici import SICIExperiment
from run_ebm import load_ncep_data

# Ensure figures directory exists
fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10


## 1. Baseline Model & Analytical Verification

We solve the numerical model on a staggered grid with $N=200$ points and standard parameters:
- $S_0 = 1370\ \mathrm{W/m^2}$
- $\alpha = 0.33$
- $A = 204\ \mathrm{W/m^2},\ B = 1.9\ \mathrm{W/(m^2\,^\circ C)}$
- $D = 0.42$

We verify the numerical solution against the exact analytical Legendre polynomial formula (Section 2 in `EBM_eng3.pdf`) and compare with hemispherically symmetrized NCEP reanalysis data.


In [ ]:
# Initialize and solve the numerical model
ebm = EnergyBalanceModel(N=200, A=204.0, B=1.9, Diff=0.42, So=1370.0, alpha=0.33)
sol_num = ebm.solve()

# Exact analytical solution at the same coordinates
sol_ana = ebm.analytical_solution(slat=sol_num["sinlat"])

# Load NCEP observations
lat_ncep, sinlat_ncep, temp_ncep = load_ncep_data()

# Check accuracy
max_err = np.max(np.abs(sol_num["Temperature"] - sol_ana["Temperature"]))
print(f"Max absolute difference vs. exact analytical solution: {max_err:.3e} °C")
print(f"Global mean temperature (numerical):  {np.mean(sol_num['Temperature']):.2f} °C")
print(f"Global mean temperature (analytical): {sol_ana['T_mean']:.2f} °C")

# Plot comparison
plt.figure(figsize=(8, 5))
plt.plot(sol_ana["lat"], sol_ana["Temperature"], color="#d95f02", lw=3.0, alpha=0.9, label="Analytical (P2 Legendre)")
plt.plot(sol_num["lat"], sol_num["Temperature"], color="#1f78b4", ls="--", lw=2.0, dashes=(4, 3), label="Numerical EBM (D=0.42, alpha=0.33)")
plt.plot(lat_ncep, temp_ncep, "ko", ms=4, alpha=0.75, label="NCEP Reanalysis (symmetrized)")
plt.xlabel("Latitude (°N)")
plt.ylabel("Temperature (°C)")
plt.title("Baseline 1-D EBM vs. Exact Analytical Solution and NCEP Data")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 2. Radiative Fluxes and Diffusive Heat Transport

The model computes:
- Absorbed solar radiation: $Q_{sw}(x) = (1 - \alpha) \frac{S_0}{4} (1 - 0.492 P_2(x))$
- Outgoing thermal radiation: $Q_{lw}(x) = A + B T(x)$
- Net radiative imbalance: $Q_{net}(x) = Q_{sw}(x) - Q_{lw}(x)$
- Diffusive heat flux across latitude circles: $F(x) = (1 - x^2) D(x) \frac{dT}{dx}$

At the equator ($x=0$) and pole ($x=1$), the transport vanishes ($F=0$). Energy is transferred from the surplus tropics ($Q_{net} > 0$) to the deficit extra-tropics ($Q_{net} < 0$).


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Radiative energy balance components
ax1.plot(sol_num["lat"], sol_num["Qsw"], "r-", lw=2, label=r"$Q_{sw}$ (Absorbed solar)")
ax1.plot(sol_num["lat"], sol_num["Qlw"], "b-", lw=2, label=r"$Q_{lw}$ (Outgoing thermal)")
ax1.plot(sol_num["lat"], sol_num["Qsw"] - sol_num["Qlw"], "k--", lw=1.5, label=r"Net imbalance ($Q_{sw} - Q_{lw}$)")
ax1.axhline(0, color="gray", lw=0.8, ls=":")
ax1.set_xlabel("Latitude (°N)")
ax1.set_ylabel(r"Flux ($W/m^2$)")
ax1.set_title("Top-of-Atmosphere Radiative Budget")
ax1.grid(True, alpha=0.3)
ax1.legend()

# Poleward heat transport (flux at cell edges)
xe = ebm.setup_grid()["xe"]
lat_edges = np.rad2deg(np.arcsin(xe))
ax2.plot(lat_edges, sol_num["heat_transport"], "g-", lw=2)
ax2.set_xlabel("Latitude (°N)")
ax2.set_ylabel(r"Poleward Heat Transport $F(x)$ (scaled units)")
ax2.set_title(r"Diffusive Transport $F(x) = (1 - x^2) D \cdot dT/dx$")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Assignment 1: Sensitivity Experiments

Here we conduct the parameter experiments described in Section 4 of `EBM_eng3.pdf`:
1. **Diffusivity Sensitivity:** $D \in \{0, 0.1, 0.42, 0.5, 1.0, 10.0\}$. For $D=0$, no lateral heat transport occurs and equator-to-pole temperature contrast is maximized. As $D \to \infty$, temperature becomes uniform. Notice that global mean temperature remains unchanged because $\int_0^1 P_2(x) dx = 0$.
2. **Variable Albedo:** $\alpha(x) = (0.315 + 0.3615 x^2) / f$ (Fanning & Weaver 1996).
3. **Variable Diffusivity:** Lindzen & Farrell (1977) parameterization representing enhanced Hadley-cell mixing:
   $$D(x) = 0.45 \left[1 + 9 \exp\left(-\left(\frac{x}{\sin 30^\circ}\right)^6\right)\right]$$


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Panel 1: Baseline vs NCEP
ax = axes[0, 0]
ax.plot(sol_ana["lat"], sol_ana["Temperature"], color="#d95f02", lw=3.0, alpha=0.9, label="Analytical (P2 Legendre)")
ax.plot(sol_num["lat"], sol_num["Temperature"], color="#1f78b4", ls="--", lw=2.0, dashes=(4, 3), label="Numerical (D=0.42, alpha=0.33)")
ax.plot(lat_ncep, temp_ncep, "ko", ms=4, alpha=0.75, label="NCEP Reanalysis (sym)")
ax.set_xlabel("Latitude (°N)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("1. Baseline vs. Observations")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left", fontsize=8.5)

# Panel 2: Varying D
ax = axes[0, 1]
diff_values = [0.0, 0.1, 0.42, 0.5, 1.0, 10.0]
colors = plt.cm.viridis(np.linspace(0, 1, len(diff_values)))
for d_val, c in zip(diff_values, colors):
    sol_d = ebm.solve(diffusion=d_val)
    label = f"D = {d_val}" if d_val > 0 else "D = 0 (no transport)"
    ax.plot(sol_d["lat"], sol_d["Temperature"], label=label, color=c, lw=1.7)
ax.plot(lat_ncep, temp_ncep, "k.", ms=3, label="NCEP")
ax.set_xlabel("Latitude (°N)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("2. Sensitivity to Diffusivity D")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left", fontsize=8)

# Panel 3: Variable Albedo
ax = axes[1, 0]
f_values = [1.0, 1.1, 1.2, 1.3]
for f in f_values:
    sol_alb = ebm.solve(albedo=lambda x, factor=f: (0.315 + 0.3615 * x**2) / factor)
    ax.plot(sol_alb["lat"], sol_alb["Temperature"], lw=1.7, label=f"f = {f}")
ax.plot(lat_ncep, temp_ncep, "ro", ms=4, alpha=0.8, label="NCEP")
ax.set_xlabel("Latitude (°N)")
ax.set_ylabel("Temperature (°C)")
ax.set_title(r"3. Variable Albedo: $\alpha(x) = (0.315 + 0.3615 x^2) / f$")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left", fontsize=9)

# Panel 4: Lindzen & Farrell (1977) D(x)
ax = axes[1, 1]
sin30 = np.sin(np.deg2rad(30.0))
d_lf = lambda x_inner: 0.45 * (1.0 + 9.0 * np.exp(-((x_inner / sin30) ** 6)))
sol_lf = ebm.solve(diffusion=d_lf)
sol_lf_tuned = ebm.solve(diffusion=d_lf, albedo=lambda x: (0.315 + 0.3615 * x**2) / 1.15)

ax.plot(sol_num["lat"], sol_num["Temperature"], "k--", lw=1.5, label="Constant D=0.42")
ax.plot(sol_lf["lat"], sol_lf["Temperature"], "b-", lw=1.8, label="Lindzen-Farrell D(x)")
ax.plot(sol_lf_tuned["lat"], sol_lf_tuned["Temperature"], "g-", lw=2, label="Lindzen-Farrell D(x) + alb(x)")
ax.plot(lat_ncep, temp_ncep, "ro", ms=4, alpha=0.8, label="NCEP")
ax.set_xlabel("Latitude (°N)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("4. Lindzen & Farrell (1977) Variable D(x)")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left", fontsize=8.5)

plt.tight_layout()
plt.savefig("figures/ebm_assignment1_results.png", dpi=180)
plt.show()


## 4. Assignment 2: Small Ice Cap Instability (SICI)

Now we couple the temperature-albedo feedback:
$$\alpha(T) = \begin{cases} 0.60 & \text{if } T < -10^\circ\mathrm{C} \\ 0.33 & \text{otherwise} \end{cases}$$

Because $\alpha$ depends on $T$ and $T$ depends on $\alpha$, the equilibrium state is found iteratively:
$$\mathbf{T}^{(k+1)} = \mathbf{K}^{-1} \left(\mathbf{A} - \mathbf{Q}_{sw}(\alpha(\mathbf{T}^{(k)}))\right)$$

Sweeping the solar constant $S_0$ reveals a **hysteresis loop** with multiple equilibrium states:
- **Snowball collapse:** Decreasing $S_0$ from modern ($1370\ \mathrm{W/m^2}$) leads to a critical threshold where ice spreads to the equator.
- **Deglaciation:** Once frozen, escaping the snowball state requires raising $S_0$ to much higher values due to high planetary albedo.


In [ ]:
# Run the SICI experiment
exp = SICIExperiment(N=200, A=204.0, B=1.9, Diff=0.42)
results = exp.run_sweep(n_up=40, n_down=71, n_warmup=250)

# Plot the bifurcation diagram
So_up, T_up = results["branch_up"]
So_down, T_down = results["branch_down"]
So_up2, T_up2 = results["branch_warmup"]

plt.figure(figsize=(9, 6))
plt.plot(So_up2, T_up2, "k-", lw=1.8, label="Snowball branch (warming up)")
plt.plot(So_up, T_up, "r-", lw=2.0, label="Warm branch (warming up)")
plt.plot(So_down, T_down, "g-", lw=2.0, label="Warm branch (cooling down)")

plt.axvline(1370.0, color="gray", linestyle=":", lw=1.2, label=r"Modern $S_0 = 1370\ \mathrm{W/m^2}$")
plt.axhline(-10.0, color="blue", linestyle="--", lw=1.0, alpha=0.6, label=r"$T_c = -10\ ^\circ\mathrm{C}$")

plt.xlabel(r"Solar Constant $S_0\ (\mathrm{W/m^2})$")
plt.ylabel(r"Global Mean Temperature $(^\circ\mathrm{C})$")
plt.title("Small Ice Cap Instability (SICI) Hysteresis Loop
(1-D EBM with Temperature-Albedo Feedback)")
plt.grid(True, alpha=0.3)
plt.legend(loc="upper left")
plt.tight_layout()
plt.savefig("figures/ebm_sici_bifurcation.png", dpi=180)
plt.show()
